# Task 1: Text Cleaning
We explore the train data only.
The test data is cleaned at the end with the same function.

   ## 1. Load the data

In [1]:
import pandas as pd
import re
from collections import Counter
train = pd.read_csv('../data/train.csv')
train.head()

,label,text
0,ham,"Sir, hope your day is going smoothly. i really..."
1,ham,Yup having my lunch buffet now.. U eat already?
2,ham,"Oh... Icic... K lor, den meet other day..."
3,ham,Yeah no probs - last night is obviously catchi...
4,ham,on a Tuesday night r u 4 real


## 2. Look at the messages

In [2]:
ham = train[train['label'] == 'ham']
spam = train[train['label'] == 'spam']
print('HAM')
for text in ham['text'].head(5):
    print(text)
    print()
print('SPAM')
for text in spam['text'].head(5):
    print(text)
    print()

HAM
Sir, hope your day is going smoothly. i really hoped i wont have to bother you about this. I have some bills that i can't settle this month. I am out of all extra cash. I know this is a challenging time for you also but i have to let you know.

Yup having my lunch buffet now.. U eat already?

Oh... Icic... K lor, den meet other day...

Yeah no probs - last night is obviously catching up with you... Speak soon 

on a Tuesday night r u 4 real

SPAM
thesmszone.com lets you send free anonymous and masked messages..im sending this message from there..do you see the potential for abuse???

Wanna get laid 2nite? Want real Dogging locations sent direct to ur mobile? Join the UK's largest Dogging Network. Txt PARK to 69696 now! Nyt. ec2a. 3lp £1.50/msg

Welcome to Select, an O2 service with added benefits. You can now call our specially trained advisors FREE from your mobile by dialling 402.

New TEXTBUDDY Chat 2 horny guys in ur area 4 just 25p Free 2 receive Search postcode or at gaytextb

## 3. Symbols in the data

In [3]:
all_text = ' '.join(train['text'])
symbols = re.findall(r'[^a-zA-Z0-9\s]', all_text)
counts = Counter(symbols)
print('Different symbols:', len(counts))
counts.most_common(25)

Different symbols: 50


[('.', 8094),
 ("'", 1410),
 (',', 1380),
 ('?', 1193),
 ('!', 1021),
 ('&', 619),
 (':', 525),
 (';', 519),
 ('-', 415),
 (')', 370),
 ('/', 316),
 ('*', 235),
 ('£', 222),
 ('"', 216),
 ('#', 198),
 ('+', 94),
 ('(', 88),
 ('ü', 88),
 ('Ü', 38),
 ('=', 33),
 ('‘', 32),
 ('\x92', 30),
 ('|', 29),
 ('@', 28),
 ('>', 27)]

In [4]:
for s in ['$', '€', '¥', '&lt;', '&gt;', '&amp;', 'http', 'www', '.com']:
    print(s, all_text.count(s))

$ 22
€ 0
¥ 0
&lt; 217
&gt; 216
&amp; 43
http 17
www 70
.com 54


## 4. Ham vs spam

In [5]:
train['label'].value_counts()

label
ham     3614
spam     522
Name: count, dtype: int64

In [6]:
for w in ['&lt;#&gt;', '$', '£', 'www', 'http', ':)', ':-)', ':(', ';)', ':D']:
    has = train['text'].str.contains(w, regex=False)
    print(w, train[has]['label'].value_counts().to_dict())

&lt;#&gt; {'ham': 161}
$ {'ham': 13, 'spam': 4}
£ {'spam': 175, 'ham': 3}
www {'spam': 68, 'ham': 2}
http {'spam': 17}
:) {'ham': 159}
:-) {'ham': 66}
:( {'ham': 10}
;) {'ham': 6}
:D {'ham': 2}


In [7]:
for p in [r'\d', r'\d{5,}', r'\d{10,}']:
    has = train['text'].str.contains(p)
    print(p, train[has]['label'].value_counts().to_dict())

\d {'ham': 590, 'spam': 489}
\d{5,} {'spam': 395, 'ham': 3}
\d{10,} {'spam': 276, 'ham': 1}


In [8]:
train['length'] = train['text'].str.len()
train['caps'] = train['text'].str.count(r'\b[A-Z]{2,}\b')
train['excl'] = train['text'].str.count('!')
train.groupby('label')[['length', 'caps', 'excl']].mean()

,length,caps,excl
label,,,
ham,70.966796,0.41782,0.182346
spam,137.741379,1.95977,0.693487


## 5. Cleaning function
Rules, in this order:
- faces :) and :( become happyface and sadface
- lowercase
- the hidden number tag becomes hiddennumber
- links become url
- money becomes pounds or dollars
- numbers with 5 digits or more become longnumber
- apostrophes are removed
- other symbols become spaces

In [9]:
def clean(text):
    text = re.sub(r'[:;]-?\)', ' happyface ', text)
    text = re.sub(r':-?D\b', ' happyface ', text)
    text = re.sub(r':-?\(', ' sadface ', text)
    text = text.lower()
    text = text.replace('&lt;#&gt;', ' hiddennumber ')
    text = re.sub(r'&(lt|gt|amp);', ' ', text)
    text = re.sub(r'(http\S*|www\S*|\S+\.com\S*)', ' url ', text)
    text = re.sub(r'£\s?\d[\d,]*(\.\d+)?', ' pounds ', text)
    text = re.sub(r'\$\s?\d[\d,]*(\.\d+)?', ' dollars ', text)
    text = re.sub(r'\d{5,}', ' longnumber ', text)
    for ch in ["'", '‘', '’', '\x92']:
        text = text.replace(ch, '')
    text = re.sub(r'[^a-z0-9 ]', ' ', text)
    return ' '.join(text.split())

In [10]:
print(spam['text'].iloc[1])
print(clean(spam['text'].iloc[1]))

Wanna get laid 2nite? Want real Dogging locations sent direct to ur mobile? Join the UK's largest Dogging Network. Txt PARK to 69696 now! Nyt. ec2a. 3lp £1.50/msg
wanna get laid 2nite want real dogging locations sent direct to ur mobile join the uks largest dogging network txt park to longnumber now nyt ec2a 3lp pounds msg


## 6. Apply to train and test, then save

In [11]:
test = pd.read_csv('../data/test.csv')
for df in [train, test]:
    df['length'] = df['text'].str.len()
    df['caps'] = df['text'].str.count(r'\b[A-Z]{2,}\b')
    df['excl'] = df['text'].str.count('!')
    df['clean_text'] = df['text'].apply(clean)
print('Empty in train:', (train['clean_text'] == '').sum())
print('Empty in test:', (test['clean_text'] == '').sum())
train.to_csv('../data/train_clean.csv', index=False)
test.to_csv('../data/test_clean.csv', index=False)
print(train.shape, test.shape)

Empty in train: 0
Empty in test: 0
(4136, 6) (1035, 6)


## 7. Checks

In [12]:
overlap = set(train['clean_text']) & set(test['clean_text'])
print('Same clean text in train and test:', len(overlap))
print('Repeated clean texts in train:', train['clean_text'].duplicated().sum())
conflict = train.groupby('clean_text')['label'].nunique().gt(1).sum()
print('Same clean text, different labels:', conflict)
raw_words = set(' '.join(train['text']).lower().split())
clean_words = set(' '.join(train['clean_text']).split())
print('Words before:', len(raw_words), '| after:', len(clean_words))

Same clean text in train and test: 18
Repeated clean texts in train: 39
Same clean text, different labels: 0
Words before: 11794 | after: 7314


## 8. Before and after

In [13]:
ids = list(ham.index[:2]) + list(spam.index[:2])
ids.append(train[train['text'].str.contains(':)', regex=False)].index[0])
ids.append(train[train['text'].str.contains('&lt;#&gt;', regex=False)].index[0])
for i in ids:
    print('BEFORE:', train['text'][i])
    print('AFTER :', train['clean_text'][i])
    print()

BEFORE: Sir, hope your day is going smoothly. i really hoped i wont have to bother you about this. I have some bills that i can't settle this month. I am out of all extra cash. I know this is a challenging time for you also but i have to let you know.
AFTER : sir hope your day is going smoothly i really hoped i wont have to bother you about this i have some bills that i cant settle this month i am out of all extra cash i know this is a challenging time for you also but i have to let you know

BEFORE: Yup having my lunch buffet now.. U eat already?
AFTER : yup having my lunch buffet now u eat already

BEFORE: thesmszone.com lets you send free anonymous and masked messages..im sending this message from there..do you see the potential for abuse???
AFTER : url lets you send free anonymous and masked messages im sending this message from there do you see the potential for abuse

BEFORE: Wanna get laid 2nite? Want real Dogging locations sent direct to ur mobile? Join the UK's largest Dogging